# El Oráculo inmobiliario de California

En este notebook se van a entrenar 2 modelos predictivos básicos de Machine Learning y consta de misiones:

**Misión 1:** Predecir el Precio del Sueño Californiano.

Predecir el precio de las casas de California basándonos en el *dataset* `housing.csv`, utilizando *regresión lineal*.

**Misión 2:** Detectar al Cliente Traidor.

Predecir qué clientes se van, utilizando *regresión logística*.

## Misión 1: Predecir el Precio del Sueño Californiano

### Recordatorio del EDA

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

df = pd.read_csv("housing.csv")
df.shape

(20640, 10)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [3]:
df.corr(numeric_only=True)["median_house_value"].sort_values(ascending=False)

median_house_value    1.000000
median_income         0.688075
total_rooms           0.134153
housing_median_age    0.105623
households            0.065843
total_bedrooms        0.049686
population           -0.024650
longitude            -0.045967
latitude             -0.144160
Name: median_house_value, dtype: float64

In [4]:
df.groupby("ocean_proximity")["median_house_value"].median()

ocean_proximity
<1H OCEAN     214850.0
INLAND        108500.0
ISLAND        414700.0
NEAR BAY      233800.0
NEAR OCEAN    229450.0
Name: median_house_value, dtype: float64

In [5]:
df["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [6]:
df["median_house_value"].value_counts().head()

median_house_value
500001.0    965
137500.0    122
162500.0    117
112500.0    103
187500.0     93
Name: count, dtype: int64

In [7]:
df["housing_median_age"].value_counts().head()

housing_median_age
52.0    1273
36.0     862
35.0     824
16.0     771
17.0     698
Name: count, dtype: int64

In [8]:
df["median_income"].value_counts().head()

median_income
3.1250     49
15.0001    49
2.8750     46
2.6250     44
4.1250     44
Name: count, dtype: int64

In [9]:
df["median_income"].max()

np.float64(15.0001)

En el análisis anterior del dataset encontramos varios hallazgos importantes. Los números de abajo salen del dataset completo, antes de limpiarlo.

- **Ingreso mediano:** La variable que más se asocia con el precio de las zonas es `median_income`, con r = 0,688.
- **Ubicación:** El precio mediano más bajo es el de las zonas `INLAND` (108.500) y el más alto es el de `ISLAND` (414.700), aunque `ISLAND` tiene solo 5 zonas.
- **Topes:** hay valores que se repiten de forma sospechosa: 500.001 en `median_house_value` (965 zonas), 52 en `housing_median_age` (1.273 zonas) y 15,0001 en `median_income` (49 zonas), que parece un tope.
- **Nulos:** `total_bedrooms` tiene 207 zonas con valores nulos.


### Preparación de datos

Se decidió **conservar** las 207 zonas con valores nulos en `total_bedrooms`. Esa columna tiene una correlación de 0,0497 con el precio `median_house_value`, mientras que esas zonas tienen completo `median_income`, la variable con la correlación más alta con el precio que es 0,688. Los nulos se rellenarán después de la división train/test.

In [10]:
pd.get_dummies(df["ocean_proximity"]).head()

,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
0,False,False,False,True,False
1,False,False,False,True,False
2,False,False,False,True,False
3,False,False,False,True,False
4,False,False,False,True,False


In [13]:
ubicacion = pd.get_dummies(df["ocean_proximity"]).drop(columns=["INLAND"])
ubicacion.sum()

<1H OCEAN     9136
ISLAND           5
NEAR BAY      2290
NEAR OCEAN    2658
dtype: int64

Ya que `ocean_proximity` es una columna de categorías, el modelo de regresión lineal no puede usarla directamente, entonces cada una de sus 5 categorías se convirtió en su propia columna booleana, que el modelo sí puede utilizar.

Se eliminó la columna `INLAND`, que es la categoría con la mediana de precio más baja, 108.500. Sus zonas, 6.551, se reconocen porque las otras cuatro columnas valen `False`. Se quita la columna porque con las cinco, una es redundante y los coeficientes no tendrían un valor único. Con `INLAND` como referencia, el coeficiente de cada categoría se lee respecto a ella.

In [16]:
df_california = pd.concat([df.drop(columns=["ocean_proximity"]), ubicacion], axis=1)
df_california.shape

(20640, 13)

El DataFrame `df_california` conserva las 20.640 zonas del DataFrame original y ya no tiene columnas de texto: se quitó `ocean_proximity` y se agregaron las 4 columnas de `ubicacion`, por lo que ahora tiene 13 columnas en total. Todavía quedan las 207 zonas con `total_bedrooms` nulo, que se rellenan después de la división train/test.

### División train/test

In [17]:
X = df_california.drop(columns=["median_house_value"])
y = df_california["median_house_value"]
X.shape, y.shape

((20640, 12), (20640,))

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=36)
X_train.shape, X_test.shape

((16512, 12), (4128, 12))